# DCT Laboratory — Volume II, Chapter 9
## Stochastic Enterprise Optimization
**Seed `26209`** · Companion to Chapter 9 and **AXIOM Lab 2.9** (module **AXIOM-09**) · Mirrored in `DCT_V2_Ch09_Lab.xlsx`

This laboratory reproduces **Worked Example 9.1, *Can a Rule Replace the Committed Delay?***, with the book's
declarations, and works the five steps of AXIOM Lab 2.9: estimate and check, design on a sample, compare rules,
price information, and steer with outside events. The liquidity model is Volume I, Chapter 8's law for the digital
program on the book's 100,000 paths (seed 8); every answer is judged on 100,000 fresh paths drawn with the lab's
seed, 26209, which is the seed of the book's fresh paths. Its second model is Chapter 8's hiring loop with human
resources' record of departures and yields. Every Meridian number below is the book's (Table 9.3, Figures 9.1–9.3,
Online Appendix 9.B.2–9.B.3); the final cell checks them. Money is in \$ million.

## The declarations (Worked Example 9.1, Step 1)

The law is Volume I, Chapter 8's for the digital program, quarter by quarter over three years. Planned liquidity,
\$620 million today, follows the Chapter 1 plan, whose draw dips to \$482 million in quarter 7. Execution and market
surprises $E$ move it, $E' = 0.6E + w$ with $w$ of standard deviation \$50 million a quarter: 60 percent of a
surprise carries into the next quarter. Funding windows close at random, about once a decade (0.10 a year), each
removing a gamma-distributed amount of cash (mean \$120 million, standard deviation \$50 million) that is never
recovered. The board's appetite: at most a 5 percent chance that liquidity falls below the \$350 million floor in
some quarter of the program's 12.

Two levers are decided today, priced as in Volume I, Worked Example 15.1. A standby **line** $l$ covers up to $l$
of each closure, for fees of $0.03\,l$ over the program (the treasury). A **committed delay** of $\phi$ quarters
makes the draw $28 - 4\phi$ index points deep instead of 28, at \$8 million a quarter (the transformation office);
a quarter of it lifts liquidity by up to \$24.8 million, at the deepest point of the draw. The draw's shape is
$S_k = (t/2)e^{1 - t/2}$ in quarter $k$, $t = k/4$ years, equal to 1 at two years. The transformation office now
declares a third lever: a **slowdown** of $\phi'$ quarters, decided at the end of quarter $k$, cuts the rest of
the draw as a committed delay of the same length would have cut it, adding $4\phi'(S_t - S_k)$ points to liquidity
in each later quarter $t$, and delays the platform by $\phi'(1 - S_k)$ quarters, its length times the share of
the draw still ahead, at the same \$8 million a quarter. Rules read cash only at the end of a quarter.

The book's paths are drawn exactly as Volume I draws them (seed 8, Online Appendix 1.B.4): each quarter the
execution shocks, the closure counts, then the closure sizes path by path. The fresh paths are the same law
drawn with seed 26209.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26209        # the lab's seed: the fresh paths every answer is judged on (the book's fresh paths)
BOOK_SEED = 8       # Volume I, Chapter 8's stream, on which the book designs every package and rule

N, Q = 100_000, 12
FLOOR = 350.0                               # the board's floor ($m)
A_E, SD_W = 0.6, 50.0                       # execution and market: E' = 0.6 E + w, w with sd $50m a quarter
LAM, J_MEAN, J_SD = 0.10, 120.0, 50.0       # funding-window closures: 0.10 a year, gamma sizes, never recovered
DEPTH0, DPQ = 28.0, 4.0                     # the draw's depth (index points); a quarter of delay: 4 points shallower
FEE, VQ, ALPHA = 0.03, 8.0, 0.05            # line fees over the program; a quarter of delay ($m); the appetite

def relax(x0, x_inf, k, t):
    return x_inf + (x0 - x_inf) * np.exp(-k * t)

def trough(depth, tau, t):
    return depth * (t / tau) * np.exp(1 - t / tau)

TQ = np.arange(1, Q + 1) / 4                # the twelve quarters, in years
BASE = 6.2 * relax(100, 112, 0.35, TQ)      # planned liquidity without the draw ($m; index 100 = $620 million)
S = trough(1.0, 2.0, TQ)                    # the draw's shape, 1 at two years
SHAPE = 6.2 * S                             # the draw, $m per index point of depth
AHEAD = 1 - S                               # share of the draw still ahead after quarter k
PLAN = BASE - DEPTH0 * SHAPE                # the plan's liquidity, no levers

def draw_paths(seed, n=N, a_e=A_E):
    # Volume I's random stream in the book's order: each quarter the execution shocks, the closure counts, then
    # the closure sizes path by path. a_e recombines the same shocks with another persistence (Step 5's test).
    rng = np.random.default_rng(seed)
    W, closures = np.empty((n, Q)), []
    for k in range(Q):
        W[:, k] = rng.normal(0.0, SD_W, n)
        count = rng.poisson(LAM / 4, n)
        size = rng.gamma((J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN, count.sum())
        closures.append((np.repeat(np.arange(n), count), size))
    E, e = np.empty((n, Q)), np.zeros(n)
    for k in range(Q):
        e = a_e * e + W[:, k]
        E[:, k] = e
    return E, closures

class Paths:
    # A line l covers up to l of each closure; phi quarters of committed delay make the draw 28 - 4 phi deep.
    # A path stays above the floor exactly when the draw is no deeper than its critical depth, the least over the
    # quarters of (liquidity without the draw - floor) / (the draw per point), so chances are exact counts.
    def __init__(self, seed, n=N, a_e=A_E):
        self.seed, self.n = seed, n
        self.E, self.closures = draw_paths(seed, n, a_e)
        self.closure_ahead = np.zeros(n, bool)
        for path, size in self.closures:
            self.closure_ahead[path] = True
        self.memo = {}
    def lost(self, line):                   # cumulative closures not covered by the line ($m), quarter by quarter
        out, cum = np.empty((self.n, Q)), np.zeros(self.n)
        for k, (path, size) in enumerate(self.closures):
            cum = cum + np.bincount(path, np.maximum(size - line, 0.0), self.n)
            out[:, k] = cum
        return out
    def liq(self, line, phi):               # liquidity ($m), path by path and quarter by quarter
        return BASE - (DEPTH0 - DPQ * phi) * SHAPE + self.E - self.lost(line)
    def crit(self, line):
        key = round(float(line), 6)
        if key not in self.memo:
            self.memo[key] = ((BASE + self.E - self.lost(line) - FLOOR) / SHAPE).min(axis=1)
        return self.memo[key]
    def odds(self, line, phi):              # the chance of a breach in some quarter
        return float(np.mean(self.crit(line) < DEPTH0 - DPQ * phi))
    def delay_for(self, line, p=ALPHA):     # least delay meeting p: the depth it allows is a quantile of the critical depths
        c = np.sort(self.crit(line))
        d = min(DEPTH0, c[int(np.floor(p * self.n + 1e-9))])
        return np.nan if d < 0 else (DEPTH0 - d) / DPQ

def cost(line, phi):
    return FEE * line + VQ * phi

BOOK = Paths(BOOK_SEED)                     # the book's 100,000 paths
FRESH = Paths(SEED)                         # 100,000 fresh paths: the same law, the lab's seed
LIM = int(np.floor(ALPHA * N + 1e-9))       # 5,000 paths may breach
print("planned liquidity by quarter ($m):", " ".join(f"{x:.0f}" for x in PLAN),
      f"-> low point {PLAN.min():.0f} in quarter {PLAN.argmin() + 1}")
print("share of the draw still ahead after quarters 1-6 (%):", " ".join(f"{100 * x:.1f}" for x in AHEAD[:6]))
print(f"a quarter of committed delay lifts the deepest point of the draw by 4 x 6.2 = {DPQ * SHAPE.max():.1f}")
print(f"the book's paths: {BOOK.closure_ahead.mean():.1%} see a closure during the program; "
      f"the fresh paths: {FRESH.closure_ahead.mean():.1%}")

## Panel 1 — Estimate and check: one quarter at a time, and jointly (Proposition 9.2(i); Figure 9.1; Section 9.1: 24.6%, 8.4%, 1.7%, 11.2%)

AXIOM Lab 2.9, step 1. With no levers, the average path clears the floor with a \$460 million trough, yet
24.6 percent of the paths breach it: the average of the outcomes is not the outcome of the average (Section 9.1).
Strip out the closures and count the breaches that execution and market surprises alone cause, quarter by quarter
and over the program. Each quarter's chance also follows from the Gaussian law of Volume I, Theorem 8.3: $E_k$ has
mean zero and variance $50^2(1 - 0.6^{2k})/(1 - 0.6^2)$. The chance of a breach in some quarter needs the whole
path; it lies between the worst quarter's and the sum over the quarters (Proposition 9.2(i)). Then the full law
under Volume I's package, a \$170 million line and two-thirds of a quarter of delay. A chance estimated on $N$ paths
has standard error $\sqrt{p(1 - p)/N}$ (Theorem 9.3(i)).

In [ ]:
L0 = BOOK.liq(0.0, 0.0)                     # the full law, no levers
mean_path = L0.mean(axis=0)
full_any = BOOK.odds(0.0, 0.0)
noise = PLAN + BOOK.E                       # execution and market surprises only, no levers
by_q = (noise < FLOOR).mean(axis=0)
some_q = float((noise < FLOOR).any(axis=1).mean())
var_k = SD_W ** 2 * (1 - A_E ** (2 * np.arange(1, Q + 1))) / (1 - A_E ** 2)
gauss = stats.norm.cdf((FLOOR - PLAN) / np.sqrt(var_k))      # Volume I, Theorem 8.3: E_k is Gaussian
L_V1 = 170.0
PHI_V1 = BOOK.delay_for(L_V1)               # Volume I's package: $170m and the least delay that meets 5 percent
Lp = BOOK.liq(L_V1, PHI_V1)
by_q_pkg = (Lp < FLOOR).mean(axis=0)
some_pkg = float((Lp < FLOOR).any(axis=1).mean())
se5 = np.sqrt(ALPHA * (1 - ALPHA) / N)

print(f"no levers: the average path's trough is {mean_path.min():.0f} (quarter {mean_path.argmin() + 1}), above the floor; "
      f"yet {100 * full_any:.2f}% of the paths breach it")
print("\nquarter                    " + "".join(f"{k:6d}" for k in range(1, Q + 1)))
print("surprises only, simulated  " + "".join(f"{100 * x:6.2f}" for x in by_q))
print("surprises only, Gaussian   " + "".join(f"{100 * x:6.2f}" for x in gauss))
print("full law, Volume I's pkg   " + "".join(f"{100 * x:6.2f}" for x in by_q_pkg))
print(f"\nsurprises only: some quarter {100 * some_q:.2f}%; worst quarter {100 * by_q.max():.2f}% (quarter "
      f"{by_q.argmax() + 1}); sum over the quarters {100 * gauss.sum():.2f}% (Gaussian), {100 * by_q.sum():.2f}% (simulated)")
print(f"Volume I's package (line {L_V1:.0f}, delay {PHI_V1:.4f}): some quarter {100 * some_pkg:.2f}%; worst quarter "
      f"{100 * by_q_pkg.max():.2f}% (quarter {by_q_pkg.argmax() + 1}); sum over the quarters {(Lp < FLOOR).sum() / 1000:.3f}%")
print(f"standard error of a 5% chance on {N:,} paths: {100 * se5:.3f} percentage points")

qs = np.arange(1, Q + 1)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.bar(qs - 0.2, 100 * by_q, width=0.38, color="#1F4E79", label="surprises only, no levers (simulated)")
ax.plot(qs - 0.2, 100 * gauss, "o", color="#E65100", ms=4, label="surprises only: the Gaussian law")
ax.bar(qs + 0.2, 100 * by_q_pkg, width=0.38, color="#999999", label="full law under Volume I's package")
for y, col, ls, txt in ((100 * gauss.sum(), "#616161", "--", "sum over the quarters"),
                        (100 * some_q, "#B71C1C", "-", "surprises only: some quarter"),
                        (100 * some_pkg, "#1F4E79", ":", "Volume I's package: some quarter")):
    ax.axhline(y, color=col, ls=ls, lw=1.1)
    ax.text(0.45, y + 0.2, f"{txt}, {y:.1f}%", fontsize=8, color=col)
ax.set(xlim=(0.3, 12.7), ylim=(0, 12.5), xticks=qs, xlabel="quarter of the program",
       ylabel="chance of a breach (%)", title="Quarters under 2% combine to an 8.4% chance (seed 26209)")
ax.legend(frameon=False, fontsize=8, loc="center right", bbox_to_anchor=(1.0, 0.535)); plt.tight_layout(); plt.show()

## Panel 2 — Three readings of the appetite: the quantile, quarter by quarter, and CVaR (Proposition 9.2(ii)–(iii); Section 9.4; Worked Example 9.1, Step 2: \$10.44m, \$1.86m, \$18.31m)

AXIOM Lab 2.9, step 2, first part. For a line $l$, every path needs a least delay to stay above the floor, set by
its critical depth; the package's delay is the 95th percentile of those delays, the 5,001st-smallest critical depth
(the quantile form, Proposition 9.2(ii)), and the line is the cheapest on a \$5 million grid. This is Volume I's
package: the answer of the sample problem when nothing can be read. It is judged on the fresh paths
(Theorem 9.3(iv)). Read quarter by quarter instead, the appetite would require only 5 percent in each quarter
separately. The convex safe version of Proposition 9.2(iii) requires the conditional value at risk of the largest
shortfall below the floor, the average of its worst 5 percent, to be at most zero; for each line the delay is
found by bisection. The quantile itself need not be convex in the decisions: Online Appendix 9.B's example ends
the panel.

In [ ]:
def committed(P, p=ALPHA, lines=np.arange(120.0, 220.01, 5.0)):
    # the cheapest committed package meeting p on the paths P: the line on a $5m grid, the delay continuous
    best = None
    for l in lines:
        ph = P.delay_for(l, p)
        if np.isnan(ph):
            continue
        if best is None or cost(l, ph) < best[2]:
            best = (float(l), float(ph), float(cost(l, ph)))
    return best

PKG = committed(BOOK)
pkg_fresh = FRESH.odds(PKG[0], PKG[1])
z_fresh = (ALPHA - pkg_fresh) / se5
PKG_F = committed(FRESH)
print("line   delay that meets 5%   cost")
for l in (150.0, 160.0, 165.0, 170.0, 175.0, 180.0, 190.0):
    print(f"{l:5.0f} {BOOK.delay_for(l):14.4f} {cost(l, BOOK.delay_for(l)):14.4f}")
print(f"Volume I's package: line {PKG[0]:.0f}, delay {PKG[1]:.4f}, cost {PKG[2]:.4f} = fees {FEE * PKG[0]:.2f} + delay "
      f"{VQ * PKG[1]:.2f}; book's paths {100 * BOOK.odds(PKG[0], PKG[1]):.2f}%")
print(f"on the fresh paths: {100 * pkg_fresh:.3f}%, {z_fresh:.2f} standard errors below 5%: the appetite holds")
print(f"designed on the fresh paths instead: line {PKG_F[0]:.0f}, delay {PKG_F[1]:.4f}, cost {PKG_F[2]:.4f}")

def single_quarter(P, p=ALPHA, lines=np.arange(0.0, 220.01, 5.0)):
    # the appetite misread quarter by quarter: at most p in each quarter separately (a quantile of each quarter's depth)
    idx = int(np.floor(p * P.n + 1e-9))
    best = None
    for l in lines:
        ck = (BASE + P.E - P.lost(l) - FLOOR) / SHAPE
        d = min(DEPTH0, np.partition(ck, idx, axis=0)[idx, :].min())
        if d < 0:
            continue
        ph = (DEPTH0 - d) / DPQ
        if best is None or cost(l, ph) < best[2]:
            L = P.liq(l, ph)
            best = (float(l), float(ph), float(cost(l, ph)), float((L < FLOOR).mean(axis=0).max()),
                    float((L < FLOOR).any(axis=1).mean()))
    return best

def cvar_package(P, p=ALPHA, lines=np.arange(100.0, 300.01, 5.0)):
    # Proposition 9.2(iii): Y = the largest shortfall below the floor over the quarters; CVaR_p(Y) <= 0
    m = int(np.floor(p * P.n + 1e-9))
    best = None
    for l in lines:
        short0 = FLOOR - P.liq(l, 0.0)                      # shortfall with no delay, quarter by quarter
        def cvar(ph):
            Y = (short0 - DPQ * ph * SHAPE).max(axis=1)
            return float(np.partition(Y, P.n - m)[P.n - m:].mean())
        lo, hi = 0.0, DEPTH0 / DPQ
        if cvar(hi) > 0:
            continue
        if cvar(0.0) <= 0:
            hi = 0.0
        else:
            for _ in range(40):
                mid = 0.5 * (lo + hi)
                lo, hi = (lo, mid) if cvar(mid) <= 0 else (mid, hi)
        if best is None or cost(l, hi) < best[2]:
            best = (float(l), float(hi), float(cost(l, hi)), P.odds(l, hi))
    return best

SQ = single_quarter(BOOK)
sq_fresh = float((FRESH.liq(SQ[0], SQ[1]) < FLOOR).any(axis=1).mean())
CV = cvar_package(BOOK)
cv_fresh = FRESH.odds(CV[0], CV[1])
print(f"\nread quarter by quarter: line {SQ[0]:.0f}, delay {SQ[1]:.4f}, cost {SQ[2]:.4f}; worst quarter {100 * SQ[3]:.2f}%, "
      f"some quarter {100 * SQ[4]:.2f}% (fresh {100 * sq_fresh:.2f}%)")
print(f"CVaR, the convex safe version: line {CV[0]:.0f}, delay {CV[1]:.4f}, cost {CV[2]:.4f}; chance of a breach "
      f"{100 * CV[3]:.3f}% (fresh {100 * cv_fresh:.3f}%)")
print(f"convexity buys tractability and gives up {CV[2] - PKG[2]:.3f}")

# Online Appendix 9.B, Proposition 9.2(ii): c = z1 x1 + z2 x2, x_i = 1 with chance 0.04, else 0; alpha = 0.05
def q95(z1, z2):
    vals = np.array([0.0, z1, z2, z1 + z2])
    prob = np.array([0.96 * 0.96, 0.04 * 0.96, 0.96 * 0.04, 0.04 * 0.04])
    order = np.argsort(vals)
    return float(vals[order][np.searchsorted(np.cumsum(prob[order]), 0.95 - 1e-12)])
print(f"\na quantile need not be convex: q95 = {q95(1, 0):.1f} at (1, 0) and {q95(0, 1):.1f} at (0, 1), but "
      f"{q95(0.5, 0.5):.1f} at their midpoint, where c > 0 with chance 1 - 0.96^2 = {1 - 0.96 ** 2:.4f}")

## Panel 3 — Design on a sample: the sample is part of the answer (Theorem 9.3; Figure 9.2: 38%, \$10.19m; 87%, \$12.09m; 97%, \$10.97m)

AXIOM Lab 2.9, step 2, second part. Design the committed package on a small sample and judge it on the fresh
100,000 paths. As in Online Appendix 9.B.2, 200 independent samples of 1,000 paths (seeds 900000–900199) and 100
of 10,000 (seeds 900000–900099) are drawn from the same law; on each, the cheapest package is designed, aimed at
5 percent or at a margin below it, and its true chance of a breach is read on the fresh paths. Sample average
approximation is consistent but optimistic (Theorem 9.3(ii)–(iii)): a design fitted to its own small sample looks
cheap and buys too little. Part (iv) prices a margin $\delta$ in advance: a package whose true chance exceeds
5 percent passes a test at $5\% - \delta$ on $N$ paths with probability at most $e^{-N\kappa}$, with $\kappa$ of
Equation 9.5, against Hoeffding's $e^{-2N\delta^2}$.

In [ ]:
def study(n_small, reps, margins, seed0=900000):
    # design on reps independent samples of n_small paths (seeds seed0, seed0 + 1, ...); judge on the fresh paths
    true_p, costs = {mg: [] for mg in margins}, {mg: [] for mg in margins}
    for rep in range(reps):
        P = Paths(seed0 + rep, n=n_small)
        for mg in margins:
            design = committed(P, p=ALPHA - mg)
            if design is not None:
                true_p[mg].append(FRESH.odds(design[0], design[1]))
                costs[mg].append(design[2])
    out = {}
    for mg in margins:
        tp, cs = np.array(true_p[mg]), np.array(costs[mg])
        out[mg] = dict(meet=float(np.mean(tp <= ALPHA)), mean_true=float(tp.mean()), q90=float(np.quantile(tp, 0.9)),
                       mean_cost=float(cs.mean()), reps=len(tp), true=tp)
    return out

S1000 = study(1000, 200, (0.0, 0.01, 0.02))
S10000 = study(10000, 100, (0.0, 0.0025))
print("designed on        margin   designs   meet 5%   mean true chance   90th percentile   mean cost")
for n_, res in ((1000, S1000), (10000, S10000)):
    for mg, d in res.items():
        print(f"{n_:6,d} paths {100 * mg:10.2f} pt {d['reps']:7d} {100 * d['meet']:9.1f}% {100 * d['mean_true']:14.2f}% "
              f"{100 * d['q90']:16.2f}% {d['mean_cost']:12.2f}")
print(f"{100_000:6,d} paths (the book's package) {'':16s} fresh {100 * pkg_fresh:.2f}% at {PKG[2]:.2f}")

def margin_bound(n, delta, p=ALPHA):
    # Theorem 9.3(iv), Equation 9.5: kappa is the relative entropy of p - delta with respect to p (Chernoff)
    a = p - delta
    kappa = a * np.log(a / p) + (1 - a) * np.log((1 - a) / (1 - p))
    return dict(n=n, delta=delta, kappa=kappa, chernoff=np.exp(-n * kappa), hoeffding=np.exp(-2 * n * delta ** 2))

BOUNDS = [margin_bound(100_000, 0.0025), margin_bound(10_000, 0.0025), margin_bound(1000, 0.01), margin_bound(1000, 0.02)]
print("\npaths     margin   kappa        e^(-N kappa)   Hoeffding e^(-2N delta^2)   2 delta^2")
for b_ in BOUNDS:
    print(f"{b_['n']:7,d} {100 * b_['delta']:6.2f} pt {b_['kappa']:.4e} {b_['chernoff']:12.4f} {b_['hoeffding']:20.4f} "
          f"{2 * b_['delta'] ** 2:18.2e}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
bins = np.arange(2.0, 9.01, 0.25)
for mg, col, lab in ((0.0, "#B71C1C", "aimed at 5%"), (0.01, "#1F4E79", "aimed at 4% (a margin of one point)")):
    d = S1000[mg]
    ax.hist(100 * d["true"], bins=bins, color=col, alpha=0.55, label=f"{lab}: {100 * d['meet']:.1f}% meet 5%, "
            f"average cost \\${d['mean_cost']:.2f}m")
ax.axvline(5.0, color="#333333", lw=1)
ax.text(5.08, 40, "the appetite", fontsize=8)
ax.set(ylim=(0, 52), xlabel="true chance of a breach on the fresh paths (%)", ylabel="designs (of 200)",
       title="Designed on 1,000 paths, most packages miss the appetite (seed 26209)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Panel 4 — Compare rules: read cash once (Worked Example 9.1, Step 3; Table 9.3; Figure 9.3(a): \$10.62m to \$12.31m slowing only; \$10.23m committing and slowing at quarter 3)

AXIOM Lab 2.9, step 3, first part. A one-review rule reads cash at the end of quarter $k$ and slows the build by
$\phi'$ quarters on the paths whose cash is lowest — as few as the appetite allows — with or without a committed
delay. For a line, a committed delay, a quarter and a slowdown, sort the paths by cash at the review: slowing the
$m$ lowest changes which paths breach, the least $m$ that leaves at most 5,000 breaches sets the share slowed, and
the trigger lies halfway between the last path slowed and the first not slowed. The book searches lines
\$150–190 million, committed delays 0–0.7 quarter in steps of 0.05 and slowdowns 0.25–7 in steps of 0.25 (Online
Appendix 9.B.2). Here the rules that only slow are searched on that whole grid for every quarter; the rules that
commit and slow are evaluated at the book's answer for each quarter, and searched on the whole grid for quarter 3,
the best quarter to read. Every rule is then run on the fresh paths.

In [ ]:
def slow_interval(L, k):
    # for a slowdown decided at the end of quarter k: per path, the slowdowns [lo, hi] that keep quarters k+1..12 above
    d = SHAPE[k:] - SHAPE[k - 1]            # liquidity per point of slowdown in each later quarter
    gap = FLOOR - L[:, k:]
    lo, hi = np.zeros(L.shape[0]), np.full(L.shape[0], np.inf)
    for j in range(d.size):
        dj = DPQ * d[j]
        if dj > 1e-12:
            lo = np.maximum(lo, gap[:, j] / dj)
        elif dj < -1e-12:
            hi = np.minimum(hi, gap[:, j] / dj)     # past the trough a slowdown lowers liquidity: an upper bound
        else:
            lo = np.where(gap[:, j] > 0, np.inf, lo)
    return lo, hi

def one_review(P, line, base, k, slows):
    # for each slowdown: slow the fewest lowest-cash paths at the end of quarter k that keep the appetite
    lim = int(np.floor(ALPHA * P.n + 1e-9))
    L = P.liq(line, base)
    b0 = (L < FLOOR).any(axis=1)                        # breaches if not slowed
    early = (L[:, :k] < FLOOR).any(axis=1)              # breaches before the slowdown can act
    lo, hi = slow_interval(L, k)
    order = np.argsort(L[:, k - 1], kind="stable")      # lowest cash first
    cb0 = np.concatenate([[0], np.cumsum(b0[order])])
    lk = L[order, k - 1]
    rules = []
    for ph in slows:
        if base + ph * AHEAD[k - 1] > DEPTH0 / DPQ + 1e-9:
            continue
        b1 = early | ~((lo <= ph) & (ph <= hi))         # breaches if slowed
        count = np.concatenate([[0], np.cumsum(b1[order])]) + (b0.sum() - cb0)
        ok = np.nonzero(count <= lim)[0]
        if len(ok) == 0:
            continue
        m = int(ok[0])
        trig = np.inf if m == P.n else (-np.inf if m == 0 else 0.5 * (lk[m - 1] + lk[m]))
        rules.append(dict(line=float(line), base=float(base), k=k, slow=float(ph), share=m / P.n, trigger=float(trig),
                          cost=float(cost(line, base) + VQ * ph * AHEAD[k - 1] * m / P.n)))
    return rules

def run_one(P, r):
    # run a one-review rule on the paths P: (chance of a breach, expected cost, share slowed)
    L = P.liq(r["line"], r["base"])
    k = r["k"]
    hit = L[:, k - 1] < r["trigger"]
    L[hit, k:] += DPQ * r["slow"] * (SHAPE[k:] - SHAPE[k - 1])
    return float((L < FLOOR).any(axis=1).mean()), float(cost(r["line"], r["base"]) + VQ * r["slow"] * AHEAD[k - 1] * hit.mean()), float(hit.mean())

LINES_1 = np.arange(150.0, 190.01, 5.0)
BASES_1 = np.arange(0.0, 0.701, 0.05)
SLOWS_1 = np.arange(0.25, 7.001, 0.25)
SLOW_ONLY = {}
for k in range(1, 7):                       # slowing only: the book's whole grid of lines and slowdowns
    found = [r for l in LINES_1 for r in one_review(BOOK, l, 0.0, k, SLOWS_1)]
    SLOW_ONLY[k] = min(found, key=lambda r: r["cost"]) if found else None
BOOK_BOTH = {1: (165, 0.60, 0.25), 2: (165, 0.60, 1.25), 3: (165, 0.55, 2.50), 4: (165, 0.60, 4.00),
             5: (160, 0.65, 7.00), 6: (170, 0.65, 5.75)}     # the book's answers: line, committed, slowdown
BOTH = {k: one_review(BOOK, l, b, k, [s])[0] for k, (l, b, s) in BOOK_BOTH.items()}
grid3 = [r for l in LINES_1 for b in BASES_1 for r in one_review(BOOK, l, b, 3, SLOWS_1)]
BEST3 = min(grid3, key=lambda r: r["cost"])
print("review   draw     slowing only                                commit and slow")
print("quarter  ahead    line  slowdown trigger  slowed  cost        line  committed slowdown trigger  slowed  cost")
for k in range(1, 7):
    so, bo = SLOW_ONLY[k], BOTH[k]
    left = (f"{so['line']:5.0f} {so['slow']:8.2f} {so['trigger']:7.0f} {100 * so['share']:6.1f}% {so['cost']:6.2f}"
            if so else f"{'no slowdown alone keeps the appetite':41s}")
    print(f"{k:5d} {100 * AHEAD[k - 1]:6.1f}%   {left}      {bo['line']:5.0f} {bo['base']:8.2f} {bo['slow']:8.2f} "
          f"{bo['trigger']:7.0f} {100 * bo['share']:6.1f}% {bo['cost']:6.2f}")
print(f"\nquarter 3, the book's whole grid ({len(grid3):,} rules that keep the appetite): the cheapest commits "
      f"{BEST3['base']:.2f}, takes a {BEST3['line']:.0f} line and slows by {BEST3['slow']:.2f} the {100 * BEST3['share']:.1f}% "
      f"of paths whose cash is below {BEST3['trigger']:.0f}: {BEST3['cost']:.4f}, {PKG[2] - BEST3['cost']:.2f} less than the package")

ONE_FRESH = {"slow 1": run_one(FRESH, SLOW_ONLY[1]), "slow 3": run_one(FRESH, SLOW_ONLY[3]), "both 3": run_one(FRESH, BOTH[3])}
for name, (p_, c_, h_) in ONE_FRESH.items():
    print(f"on the fresh paths, {name}: chance {100 * p_:.2f}%, cost {c_:.2f}, slowed {100 * h_:.1f}%")

## Panel 5 — Compare rules: read cash every quarter (Worked Example 9.1, Step 4; Table 9.3: \$10.20m; 4.95% and \$10.21m on fresh paths)

AXIOM Lab 2.9, step 3, second part. A rule that reads cash at the end of each of the first five quarters slows
the build by $\phi'$ the first time cash falls more than a margin $m$ below the committed plan's path. For a line,
a committed delay and a slowdown, a path's first trigger comes earlier as $m$ falls, so the largest $m$ that keeps
the appetite is found by bisection. The book searches lines \$150–190 million, committed delays 0–0.7 quarter in
steps of 0.1 and slowdowns 0.25–3 in steps of 0.25; here its answer is reproduced exactly, and a reduced search
over the 27 neighbouring rules on that grid (lines \$160–170 million, committed 0.4–0.6, slowdowns 1–1.5) finds
none cheaper. The figure is Figure 9.3(a): what one reading of cash can buy, quarter by quarter.

In [ ]:
def every_review(P, line, base, slow, kmax=5):
    # the largest margin below the committed plan's path whose rule keeps the appetite, by bisection
    lim = int(np.floor(ALPHA * P.n + 1e-9))
    L = P.liq(line, base)
    plan_c = BASE - (DEPTH0 - DPQ * base) * SHAPE
    M = np.maximum.accumulate(plan_c[None, :kmax] - L[:, :kmax], axis=1)    # running largest shortfall below plan
    b_none = (L < FLOOR).any(axis=1)
    b_at = []
    for k in range(1, kmax + 1):            # breaches if first slowed at the end of quarter k
        lo, hi = slow_interval(L, k)
        b_at.append((L[:, :k] < FLOOR).any(axis=1) | ~((lo <= slow) & (slow <= hi)))
    b_at = np.stack(b_at + [b_none], axis=1)
    rows = np.arange(P.n)
    def outcome(c):
        trig = M > c
        first = np.where(trig.any(axis=1), trig.argmax(axis=1), kmax)
        return int(b_at[rows, first].sum()), first
    if outcome(1e12)[0] <= lim:
        return dict(line=line, base=base, slow=slow, margin=np.inf, share=0.0, cost=float(cost(line, base)))
    if outcome(-1e12)[0] > lim:
        return None
    lo_c, hi_c = -1e6, 1e6
    for _ in range(50):
        mid = 0.5 * (lo_c + hi_c)
        lo_c, hi_c = (mid, hi_c) if outcome(mid)[0] <= lim else (lo_c, mid)
    cnt, first = outcome(lo_c)
    ahead = np.where(first < kmax, AHEAD[np.minimum(first, kmax - 1)], 0.0)
    return dict(line=float(line), base=float(base), slow=float(slow), margin=float(lo_c), share=float((first < kmax).mean()),
                cost=float(cost(line, base) + VQ * slow * ahead.mean()), p=cnt / P.n)

def run_every(P, line, base, slow, margin, kmax=5):
    # run the rule on the paths P: (chance of a breach, expected cost, share slowed, quarter first slowed)
    L = P.liq(line, base)
    trig = BASE - (DEPTH0 - DPQ * base) * SHAPE - margin
    done, when = np.zeros(P.n, bool), np.zeros(P.n, int)
    for k in range(1, kmax + 1):
        hit = ~done & (L[:, k - 1] < trig[k - 1])
        L[np.ix_(hit, np.arange(k, Q))] += DPQ * slow * (SHAPE[k:] - SHAPE[k - 1])
        when[hit] = k
        done |= hit
    ahead = np.where(done, AHEAD[np.maximum(when, 1) - 1], 0.0)
    return float((L < FLOOR).any(axis=1).mean()), float(cost(line, base) + VQ * slow * ahead.mean()), float(done.mean()), when

EVERY = every_review(BOOK, 165.0, 0.5, 1.25)
ev_book = run_every(BOOK, 165.0, 0.5, 1.25, EVERY["margin"])
ev_fresh = run_every(FRESH, 165.0, 0.5, 1.25, EVERY["margin"])
slowed_by_q = [float(np.mean(ev_book[3] == k)) for k in range(1, 6)]
contingent = (EVERY["cost"] - cost(165.0, 0.5)) / VQ
print(f"every review (quarters 1-5): line {EVERY['line']:.0f}, committed {EVERY['base']:.2f}, slow by {EVERY['slow']:.2f} the "
      f"first time cash is {EVERY['margin']:.2f} below the committed plan's path")
print(f"  book's paths: chance {100 * ev_book[0]:.2f}%, cost {EVERY['cost']:.4f}, slows {100 * EVERY['share']:.1f}% of paths "
      f"(first at quarters 1-5: {', '.join(f'{100 * x:.1f}%' for x in slowed_by_q)}), {contingent:.3f} quarter of contingent delay on average")
print(f"  saves {PKG[2] - EVERY['cost']:.3f} ({100 * (PKG[2] - EVERY['cost']) / PKG[2]:.2f}%) against Volume I's package")
print(f"  fresh paths: chance {100 * ev_fresh[0]:.2f}%, cost {ev_fresh[1]:.2f}, slows {100 * ev_fresh[2]:.1f}%")

NEAR = [every_review(BOOK, l, b, s) for l in (160.0, 165.0, 170.0) for b in (0.4, 0.5, 0.6) for s in (1.0, 1.25, 1.5)]
NEAR = [r for r in NEAR if r is not None]
near_best = min(NEAR, key=lambda r: r["cost"])
print(f"reduced search, the {len(NEAR)} neighbouring rules on the book's grid: least cost {near_best['cost']:.4f} at line "
      f"{near_best['line']:.0f}, committed {near_best['base']:.1f}, slowdown {near_best['slow']:.2f} (the book's rule)")

ks = np.arange(1, 7)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.plot(ks[:4], [SLOW_ONLY[k]["cost"] for k in range(1, 5)], "o-", color="#B71C1C", lw=1.8, label="slow only when cash is low")
ax.plot(ks, [BOTH[k]["cost"] for k in range(1, 7)], "s-", color="#1F4E79", lw=1.8, label="commit part, slow more when cash is low")
ax.axhline(PKG[2], color="#616161", lw=1.1)
ax.text(1.45, PKG[2] + 0.04, f"committed package \\${PKG[2]:.2f}m", fontsize=8, color="#616161", ha="left")
ax.axhline(EVERY["cost"], color="#E65100", lw=1.1, ls="--")
ax.text(6.3, EVERY["cost"] - 0.12, f"reading cash every quarter \\${EVERY['cost']:.2f}m", fontsize=8, color="#E65100", ha="right")
for k in range(1, 7):
    ax.text(k, 12.55, f"{100 * AHEAD[k - 1]:.0f}%", fontsize=7.5, color="#616161", ha="center")
ax.text(3.5, 12.75, "share of the draw still ahead", fontsize=7.5, color="#616161", ha="center")
ax.set(xlim=(0.6, 6.5), ylim=(9.9, 12.95), xticks=ks, xlabel="quarter whose cash is read (one reading)",
       ylabel="least cost of the appetite ($ million)", title="Reading cash saves at most \\$0.25m of \\$10.44m (seed 26209)")
ax.legend(frameon=False, fontsize=8, loc="center right", bbox_to_anchor=(1.0, 0.55)); plt.tight_layout(); plt.show()

## Panel 6 — Price information: the clairvoyant and an early warning (Proposition 9.3; Worked Example 9.1, Step 5; Table 9.3; Figure 9.3(b): \$0.44m; worth \$10.0m and \$4.1m)

AXIOM Lab 2.9, step 4. A clairvoyant sees each path's future before choosing its line and delay: each path takes
its cheapest package that keeps it above the floor (lines \$0–400 million), and the 5,000 paths whose packages cost
most are let go (Proposition 9.3(iii)). The gap to Volume I's package, the expected value of perfect information,
bounds what any reading, forecast or warning could save with these levers. A partial signal: suppose the treasury
could tell at the start whether a funding window will close during the program. The board then takes one package
for each signal and allots its appetite between them so that the overall chance stays at 5 percent
(Proposition 9.3(ii)), searching the allotment on a grid of a quarter of a point.

In [ ]:
LINES_ALL = np.arange(0.0, 400.01, 5.0)
best_pkg, with_line = np.full(N, np.inf), np.zeros(N, bool)
for l in LINES_ALL:                         # each path's cheapest package that keeps it above the floor
    c_ = BOOK.crit(l)
    need = np.where(c_ < 0, np.inf, np.maximum(0.0, (DEPTH0 - c_) / DPQ))
    cc = FEE * l + VQ * need
    better = cc < best_pkg - 1e-12
    best_pkg[better], with_line[better] = cc[better], l > 0
kept = np.argsort(best_pkg, kind="stable")[:N - LIM]       # the costliest 5 percent are let go
CLAIR = float(best_pkg[kept].sum() / N)
acting = float((best_pkg[kept] > 0).sum() / N)
lined = float(with_line[kept][best_pkg[kept] > 0].sum() / N)
EVPI = PKG[2] - CLAIR
print(f"clairvoyant: {CLAIR:.4f}; {100 * (1 - acting):.1f}% of paths need no lever, {100 * acting:.1f}% get a package, "
      f"{100 * lined:.1f}% some of the line; the costliest 5% are let go")
print(f"expected value of perfect information: {PKG[2]:.2f} - {CLAIR:.2f} = {EVPI:.2f}, the most any information could save")

ahead_ = BOOK.closure_ahead
w1 = float(ahead_.mean()); w0 = 1.0 - w1
SORTED = {g: [np.sort(BOOK.crit(l)[mask]) for l in LINES_ALL] for g, mask in ((0, ~ahead_), (1, ahead_))}
def group(g, pg):
    # the cheapest package for one signal, with chance pg of a breach given the signal
    best = None
    for l, c in zip(LINES_ALL, SORTED[g]):
        idx = int(np.floor(pg * len(c) + 1e-9))
        d = DEPTH0 if idx >= len(c) else min(DEPTH0, c[idx])
        if d < 0:
            continue
        ph = (DEPTH0 - d) / DPQ
        if best is None or cost(l, ph) < best[0]:
            best = (float(cost(l, ph)), float(l), float(ph), float(pg))
    return best
WARN = None
for p0 in np.arange(0.0, 0.0701, 0.0025):  # the appetite allotted to "no closure ahead"; the rest to "closure ahead"
    p1 = (ALPHA - w0 * p0) / w1
    if 0 <= p1 <= 1:
        g0, g1 = group(0, p0), group(1, p1)
        if g0 and g1 and (WARN is None or w0 * g0[0] + w1 * g1[0] < WARN[0]):
            WARN = (w0 * g0[0] + w1 * g1[0], g0, g1)
print(f"\nearly warning: {100 * w1:.1f}% of paths see a closure during the program")
for name, g in (("no closure ahead", WARN[1]), ("closure ahead", WARN[2])):
    print(f"  {name:17s}: line {g[1]:3.0f}, delay {g[2]:.4f}, cost {g[0]:.4f}, chance {100 * g[3]:.2f}%")
print(f"  expected cost {WARN[0]:.4f}; the warning is worth {PKG[2] - WARN[0]:.2f}, against {PKG[2] - EVERY['cost']:.2f} "
      f"for reading cash every quarter")

print("\nTable 9.3: ways of keeping Meridian's appetite, designed on the book's paths ($ million; delays in quarters)")
print(f"{'':34s} {'line':>7s} {'committed':>10s} {'slowdown':>9s} {'paths':>6s} {'cost':>7s} {'fresh odds':>11s} {'fresh cost':>11s}")
EVERY_PATH = "Every path's future (clairvoyant)"
rows93 = [("Volume I's package", PKG[0], PKG[1], None, None, PKG[2], pkg_fresh, PKG[2]),
          ("Slow only, reading quarter 1", SLOW_ONLY[1]["line"], 0.0, 1.00, SLOW_ONLY[1]["share"], SLOW_ONLY[1]["cost"],
           *ONE_FRESH["slow 1"][:2]),
          ("Slow only, reading quarter 3", SLOW_ONLY[3]["line"], 0.0, 4.00, SLOW_ONLY[3]["share"], SLOW_ONLY[3]["cost"],
           *ONE_FRESH["slow 3"][:2]),
          ("Commit and slow, quarter 3", BOTH[3]["line"], BOTH[3]["base"], BOTH[3]["slow"], BOTH[3]["share"], BOTH[3]["cost"],
           *ONE_FRESH["both 3"][:2]),
          ("Commit and slow, every quarter", EVERY["line"], EVERY["base"], EVERY["slow"], EVERY["share"], EVERY["cost"],
           ev_fresh[0], ev_fresh[1])]
for name, l_, b_, s_, sh_, c_, pf_, cf_ in rows93:
    slow_txt = "--" if s_ is None else f"{s_:.2f}"
    share_txt = "--" if sh_ is None else f"{100 * sh_:.0f}%"
    print(f"{name:34s} {l_:7.0f} {b_:10.2f} {slow_txt:>9s} {share_txt:>6s} {c_:7.2f} {100 * pf_:11.2f} {cf_:11.2f}")
print(f"{'Early warning of closures':34s} {WARN[1][1]:3.0f}, {WARN[2][1]:3.0f} {WARN[1][2]:.2f}, {WARN[2][2]:.2f} "
      f"{'':16s} {WARN[0]:7.2f}")
print(f"{EVERY_PATH:34s} {'':42s} {CLAIR:7.2f}")

## Panel 7 — Test the declared law: surprises that persist (Worked Example 9.1, Step 5: \$28.2m committed; reading cash every quarter saves \$2.3m, 8%)

The answer rests on the declared law. If 85 percent of a surprise carried into the next quarter instead of
60 percent — the same shocks, recombined — cash would say more about the quarters ahead. The committed package is
redesigned over lines \$100–300 million. The book's search for the best every-review rule under this law (lines
\$140–180 million in steps of \$10 million, committed delays 1.6–2.6, slowdowns 3–7) is not rerun here: its answer, a \$160 million line,
1.9 quarters committed and a slowdown of 5 quarters, is evaluated on the book's paths, where its margin is found
by bisection, and on the fresh paths.

In [ ]:
BOOK85, FRESH85 = Paths(BOOK_SEED, a_e=0.85), Paths(SEED, a_e=0.85)
noise85 = float(((PLAN + BOOK85.E) < FLOOR).any(axis=1).mean())
PKG85 = committed(BOOK85, lines=np.arange(100.0, 300.01, 5.0))
pkg85_fresh = FRESH85.odds(PKG85[0], PKG85[1])
EVERY85 = every_review(BOOK85, 160.0, 1.9, 5.0)
ev85_fresh = run_every(FRESH85, 160.0, 1.9, 5.0, EVERY85["margin"])
print(f"persistence 0.85: surprises alone breach in some quarter on {100 * noise85:.1f}% of paths (8.4% at 0.6)")
print(f"committed package: line {PKG85[0]:.0f}, delay {PKG85[1]:.3f}, cost {PKG85[2]:.2f} (fresh {100 * pkg85_fresh:.2f}%)")
print(f"every review, the book's rule: line 160, committed 1.9, slowdown 5.0, margin {EVERY85['margin']:.1f}, slows "
      f"{100 * EVERY85['share']:.1f}% of paths, cost {EVERY85['cost']:.2f} (fresh {100 * ev85_fresh[0]:.2f}%, {ev85_fresh[1]:.2f})")
print(f"reading cash every quarter saves {PKG85[2] - EVERY85['cost']:.2f}, {100 * (PKG85[2] - EVERY85['cost']) / PKG85[2]:.1f}% "
      f"(at 0.6: {PKG[2] - EVERY['cost']:.2f}, {100 * (PKG[2] - EVERY['cost']) / PKG[2]:.2f}%)")

## Panel 8 — Steer with outside events: Chapter 8's hiring loop (Theorem 9.2; Proposition 9.1; Online Appendix 9.B.3: gains 4.34, 4.16, 3.68, 4.52, 4.66)

AXIOM Lab 2.9, step 5. Chapter 8's hiring loop, read without the delay: the gap $d$ (bench minus target) moves as
$dd = (-ad + bv)\,dt + \sigma\,dW_1 + sv\,dW_2$, with $a = 4\ln(10/9)$, $b = 0.6$, $q = 0.0032$, $r = 0.000125$,
$\rho = \ln 1.1$ and $d_0 = -100$. Human resources' record: departures and early or late starts move the bench by a
standard deviation of 15 engineers over a year, $\sigma = 15$; a quarter's cohort yields 60 percent with a standard
deviation of 20 points, and since the yield of a quarter's approvals has standard deviation $s/\sqrt{1/4} = 2s$,
$s = 0.10$. Departures only add to the bench, so the rule is Chapter 8's and they cost $\sigma^2p/\rho$
(certainty equivalence, Theorem 9.2(ii)). An uncertain yield makes each approval cost more at the margin
(caution, Theorem 9.2(iii)): the gain solves $rbK^2 + (r\beta + qs^2)K - qb = 0$, $\beta = 2a + \rho$. Noise that
grows with the gap, $c\,d\,dW_3$, stiffens the rule (vigilance, Theorem 9.2(iv)). Guarding against a misjudged
drift, charged $h^2/(2\theta)$ a year, steers harder still: $(b^2/r - 2\theta\sigma^2)p^2 + \beta p - q = 0$, with
the smallest positive root, up to a breakdown point (Proposition 9.1(ii)).

In [ ]:
a, b, q, r = 4 * np.log(10 / 9), 0.6, 32 / 100 ** 2, 20 / 400 ** 2     # Chapter 8's declarations
rho, d0 = np.log(1.1), -100.0
SIGMA, S_YIELD = 15.0, 0.10                 # HR's record: departures (sd 15 a year); the yield of approvals
beta = 2 * a + rho

def riccati(q=q, r=r, a=a, b=b, rho=rho, c=0.0):
    # the positive root of (b^2/r) p^2 + (2a + rho - c^2) p - q = 0 and its gain b p / r
    s2, t = b * b / r, 2 * a + rho - c * c
    p_ = (-t + np.sqrt(t * t + 4 * s2 * q)) / (2 * s2)
    return p_, b * p_ / r

def cautious(s):
    # Theorem 9.2(iii): r b K^2 + (r beta + q s^2) K - q b = 0; then p = q / (b K + beta)
    al, be, ga = r * b, r * beta + q * s * s, q * b
    K_ = (-be + np.sqrt(be * be + 4 * al * ga)) / (2 * al)
    return q / (b * K_ + beta), K_

def gain_cost(K_, s=S_YIELD, sigma=SIGMA):
    # cost from 100 short of v = -K d with departures and an uncertain yield (infinite if the spread explodes)
    den = 2 * (a + b * K_) + rho - s * s * K_ * K_
    return np.inf if den <= 0 else (q + r * K_ * K_) * (d0 ** 2 + sigma ** 2 / rho) / den

def robust(theta, sigma=SIGMA):
    # Proposition 9.1(ii): the smallest positive root of (b^2/r - 2 theta sigma^2) p^2 + beta p - q = 0
    A2 = b * b / r - 2 * theta * sigma ** 2
    disc = beta ** 2 + 4 * A2 * q
    if disc < 0:
        return np.inf, np.inf
    p_ = 2 * q / (beta + np.sqrt(disc))
    return p_, b * p_ / r

p0, K0 = riccati()
drag = SIGMA ** 2 * p0 / rho
print(f"certainty equivalence: p = {p0:.8f}, K = {K0:.4f} a year, unchanged by departures; they cost sigma^2 p = "
      f"{SIGMA ** 2 * p0:.4f} a year, {drag:.3f} in present value; least cost from 100 short {p0 * d0 ** 2:.3f} + {drag:.3f} "
      f"= {p0 * d0 ** 2 + drag:.3f}")
pc1, Kc1 = cautious(S_YIELD)
pc2, Kc2 = cautious(2 * S_YIELD)
print(f"\ncaution (beta = 2a + rho = {beta:.6f}):")
for s_, p_, K_ in ((S_YIELD, pc1, Kc1), (2 * S_YIELD, pc2, Kc2)):
    best_c, ce_c = gain_cost(K_, s_), gain_cost(K0, s_)
    print(f"  s = {s_:.2f}: K = {K_:.4f}, p = {p_:.8f}, least cost p(d0^2 + sigma^2/rho) = {p_ * (d0 ** 2 + SIGMA ** 2 / rho):.3f}"
          f" (= {best_c:.3f}); the certainty-equivalent gain {K0:.4f} costs {ce_c:.3f}, {ce_c - best_c:.3f} more")
pv, Kv = riccati(c=0.5)
print(f"\nvigilance, noise c d dW with c = 0.5: p = {pv:.8f}, K = {Kv:.4f}")
THETAS = (0.0, 1.0, 2.0, 3.0, 6.0, 6.5)
ROBUST = [robust(t) for t in THETAS]
theta_flip = b * b / (2 * r * SIGMA ** 2)
theta_bar = (b * b / r + beta ** 2 / (4 * q)) / (2 * SIGMA ** 2)
print(f"\nrobustness: {'theta':8s}" + "".join(f"{t:9.1f}" for t in THETAS))
print(f"{'':12s}{'K':8s}" + "".join(f"{K_:9.3f}" for p_, K_ in ROBUST))
print(f"{'':12s}{'p x 1e4':8s}" + "".join(f"{1e4 * p_:9.3f}" for p_, K_ in ROBUST))
print(f"alpha_theta = {b * b / r:.0f} - {2 * SIGMA ** 2:.0f} theta turns negative at theta = {theta_flip:.2f}; the breakdown "
      f"point is theta-bar = {theta_bar:.4f}, where p -> 2q/beta = {2 * q / beta:.6f} and K -> {b * 2 * q / beta / r:.1f}")

## Panel 9 — The martingale audit: a value ledger with no drift (Theorem 9.1(iii); Online Appendix 9.B.3: \$11.171m at every date; the cautious gain drifts toward \$13.700m)

AXIOM Lab 2.9, step 5, continued. Book the position at the best rule's value, $pd^2 + \sigma^2p/\rho$, and add the
value earned so far: $M_t = \int_0^t e^{-\rho s}(q + rK^2)d_s^2\,ds + e^{-\rho t}(pd_t^2 + \sigma^2p/\rho)$, with
departures only. Along the best rule the ledger is a martingale: its expectation stays at the least cost. Under any
other rule it drifts; under Volume I's cautious gain, 2 a year, upward toward that rule's own cost. The expected
ledger has a closed form, since $E d_t^2 = d_0^2e^{-2gt} + \sigma^2(1 - e^{-2gt})/(2g)$ with $g = a + bK$. The lab
then runs the audit as a board would, on paths: 10,000 of its own (seed 26209), with the gap stepped exactly over
a four-hundredth of a year and the integral taken by the trapezoid rule. The paths come in antithetic pairs, each
draw also run with its signs reversed, a variance reduction (Further Reading: Glasserman): within a pair the
ledger's first-order noise cancels. These draws are the lab's, not the book's.

In [ ]:
def ledger(K_, t, sigma=SIGMA):
    # E M_t under v = -K d, booked with the best rule's value p0 d^2 + sigma^2 p0 / rho (closed form)
    g, c_ = a + b * K_, q + r * K_ * K_
    m = sigma ** 2 / (2 * g)
    integral = (d0 ** 2 - m) * (1 - np.exp(-(2 * g + rho) * t)) / (2 * g + rho) + m * (1 - np.exp(-rho * t)) / rho
    ed2 = d0 ** 2 * np.exp(-2 * g * t) + m * (1 - np.exp(-2 * g * t))
    return c_ * integral + np.exp(-rho * t) * (p0 * ed2 + sigma ** 2 * p0 / rho)

def rule_cost(K_, sigma=SIGMA):            # the limit of the ledger: the rule's own expected cost
    g, c_ = a + b * K_, q + r * K_ * K_
    m = sigma ** 2 / (2 * g)
    return c_ * ((d0 ** 2 - m) / (2 * g + rho) + m / rho)

DATES = (0.0, 1.0, 2.0, 5.0)
LEDGER = {"best": [ledger(K0, t) for t in DATES], "cautious": [ledger(2.0, t) for t in DATES]}
print("years                 " + "".join(f"{t:9.0f}" for t in DATES) + "    limit")
print(f"best rule (K = {K0:.3f})" + "".join(f"{x:9.3f}" for x in LEDGER["best"]) + f"{rule_cost(K0):9.3f}")
print(f"cautious gain (K = 2) " + "".join(f"{x:9.3f}" for x in LEDGER["cautious"]) + f"{rule_cost(2.0):9.3f}")

rng = np.random.default_rng(SEED)           # the lab's own draws (not the book's)
n_pairs, dt, T_mc = 5_000, 1 / 400, 5.0     # 10,000 paths in antithetic pairs
RULES_MC = {"best": K0, "cautious": 2.0}
state = {}
for name, K_ in RULES_MC.items():
    g, c_ = a + b * K_, q + r * K_ * K_
    state[name] = dict(c=c_, decay=np.exp(-g * dt), sd=SIGMA * np.sqrt((1 - np.exp(-2 * g * dt)) / (2 * g)),
                       d=np.full(2 * n_pairs, d0), acc=np.zeros(2 * n_pairs), f=np.full(2 * n_pairs, c_ * d0 ** 2))
MC = {name: {} for name in RULES_MC}
for i in range(1, int(round(T_mc / dt)) + 1):
    z = rng.standard_normal(n_pairs)
    z = np.concatenate([z, -z])             # each draw also with its signs reversed; the same draws for both rules
    for name, st in state.items():
        st["d"] = st["d"] * st["decay"] + st["sd"] * z          # the gap stepped exactly (Ornstein-Uhlenbeck)
        f_now = np.exp(-rho * i * dt) * st["c"] * st["d"] ** 2
        st["acc"] += 0.5 * (st["f"] + f_now) * dt                # the value earned so far, by the trapezoid rule
        st["f"] = f_now
        if i % 100 == 0:                    # quarterly marks
            led = st["acc"] + np.exp(-rho * i * dt) * (p0 * st["d"] ** 2 + SIGMA ** 2 * p0 / rho)
            pair = 0.5 * (led[:n_pairs] + led[n_pairs:])
            MC[name][i * dt] = (pair.mean(), pair.std(ddof=1) / np.sqrt(n_pairs), led.std(ddof=1) / np.sqrt(2 * n_pairs))
for name in ("best", "cautious"):
    print(f"simulated, {name:8s}: " + ", ".join(f"year {t:.0f} {MC[name][t][0]:.3f} (se {MC[name][t][1]:.4f})"
                                                 for t in (1.0, 2.0, 5.0)))
dev = max(abs(MC["best"][t][0] - ledger(K0, t)) / MC["best"][t][1] for t in MC["best"])
print(f"best rule: within {dev:.1f} standard errors of {ledger(K0, 0.0):.3f} at every quarterly mark; antithetic pairs cut "
      f"the standard error after 5 years from {MC['best'][5.0][2]:.4f} to {MC['best'][5.0][1]:.4f}")

ts = np.linspace(0, 5, 201)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for name, K_, col in (("best", K0, "#1F4E79"), ("cautious", 2.0, "#B71C1C")):
    lab = f"best rule, {K0:.2f} a year" if name == "best" else "Volume I's cautious gain, 2 a year"
    ax.plot(ts, [ledger(K_, t) for t in ts], color=col, lw=2, label=f"{lab}: expected ledger")
    tm = sorted(MC[name])[1::2]
    ax.errorbar(tm, [MC[name][t][0] for t in tm], yerr=[2 * MC[name][t][1] for t in tm], fmt="o", ms=3.5, color=col,
                capsize=2, lw=0.8, label="simulated, half-yearly (seed 26209), ± 2 standard errors" if name == "best" else None)
ax.axhline(rule_cost(2.0), color="#616161", ls=":", lw=1)
ax.text(5.0, rule_cost(2.0) + 0.06, f"the cautious gain's cost \\${rule_cost(2.0):.2f}m", fontsize=8, color="#616161", ha="right")
ax.set(xlim=(0, 5.1), ylim=(10.8, 14.1), xlabel="years", ylabel="value earned plus value booked ($ million)",
       title="Along the best rule the value ledger has no drift (seed 26209)")
ax.legend(frameon=False, fontsize=7.5, loc="center right", bbox_to_anchor=(1.0, 0.42)); plt.tight_layout(); plt.show()

## Validation — the book's numbers, and agreement with `DCT_V2_Ch09_Lab.xlsx`

Each checkpoint is a number printed in Chapter 9 (Sections 9.1–9.6, Worked Example 9.1, Table 9.3, Figures
9.1–9.3) or in Online Appendix 9.B.2–9.B.3, with a tolerance of half a unit in its last printed digit. Chances
are in percent. The workbook recomputes the ones marked *workbook* with Excel formulas and shows PASS against the
same engine values on its `Checks` sheet.

In [ ]:
pc = lambda x: 100 * x
checks = [
    # Section 9.1 and Figure 9.1 (Panel 1)
    ("no levers: the average path's trough", mean_path.min(), 460, 0.5),
    ("no levers: chance of a breach in some quarter", pc(full_any), 24.62, 5e-3),
    ("surprises only: some quarter", pc(some_q), 8.40, 5e-3),
    ("surprises only: worst quarter", pc(by_q.max()), 1.69, 5e-3),
    ("surprises only: the worst quarter is quarter 7", by_q.argmax() + 1, 7, 0),
    ("surprises only, Gaussian: quarter 7 (workbook)", pc(gauss[6]), 1.73, 5e-3),
    ("surprises only, Gaussian: quarter 12 (workbook)", pc(gauss[11]), 0.51, 5e-3),
    ("sum over the quarters, Gaussian (workbook)", pc(gauss.sum()), 11.17, 5e-3),
    ("Volume I's package: some quarter", pc(some_pkg), 5.00, 5e-3),
    ("Volume I's package: worst quarter (quarter 7)", pc(by_q_pkg.max()), 0.97, 5e-3),
    ("Volume I's package: quarter 4", pc(by_q_pkg[3]), 0.47, 5e-3),
    ("standard error of 5% on 100,000 paths, points (workbook)", pc(se5), 0.069, 5e-4),
    # Section 9.4 and Worked Example 9.1, Step 2 (Panel 2)
    ("committed package: line", PKG[0], 170, 0),
    ("committed package: delay, quarters", PKG[1], 0.6676, 5e-5),
    ("committed package: cost (workbook)", PKG[2], 10.44, 5e-3),
    ("committed package: fees (workbook)", FEE * PKG[0], 5.10, 5e-3),
    ("committed package: delay cost (workbook)", VQ * PKG[1], 5.34, 5e-3),
    ("committed package: chance on fresh paths", pc(pkg_fresh), 4.907, 5e-4),
    ("fresh chance below 5%, standard errors (workbook)", z_fresh, 1.3, 0.05),
    ("designed on the fresh paths: line", PKG_F[0], 165, 0),
    ("designed on the fresh paths: delay", PKG_F[1], 0.6658, 5e-5),
    ("designed on the fresh paths: cost (workbook)", PKG_F[2], 10.28, 5e-3),
    ("read quarter by quarter: line", SQ[0], 60, 0),
    ("read quarter by quarter: delay", SQ[1], 0.008, 5e-4),
    ("read quarter by quarter: cost (workbook)", SQ[2], 1.86, 5e-3),
    ("read quarter by quarter: some quarter", pc(SQ[4]), 16.69, 5e-3),
    ("read quarter by quarter: some quarter, fresh", pc(sq_fresh), 16.39, 5e-3),
    ("CVaR package: line", CV[0], 185, 0),
    ("CVaR package: delay", CV[1], 1.596, 5e-4),
    ("CVaR package: cost (workbook)", CV[2], 18.31, 5e-3),
    ("CVaR package: chance of a breach", pc(CV[3]), 1.9, 0.05),
    ("CVaR package: chance on fresh paths", pc(cv_fresh), 1.85, 5e-3),
    ("convexity gives up (workbook)", CV[2] - PKG[2], 7.9, 0.05),
    ("a nonconvex quantile: chance at the midpoint (workbook)", 1 - 0.96 ** 2, 0.0784, 5e-5),
    # Theorem 9.3 and Figure 9.2 (Panel 3)
    ("1,000 paths, no margin: designs meeting 5%", pc(S1000[0.0]["meet"]), 38.5, 0.05),
    ("1,000 paths, no margin: mean true chance", pc(S1000[0.0]["mean_true"]), 5.21, 5e-3),
    ("1,000 paths, no margin: 90th percentile", pc(S1000[0.0]["q90"]), 6.23, 5e-3),
    ("1,000 paths, no margin: mean cost", S1000[0.0]["mean_cost"], 10.19, 5e-3),
    ("1,000 paths, one point: designs meeting 5%", pc(S1000[0.01]["meet"]), 87.0, 0.05),
    ("1,000 paths, one point: mean true chance", pc(S1000[0.01]["mean_true"]), 4.21, 5e-3),
    ("1,000 paths, one point: 90th percentile", pc(S1000[0.01]["q90"]), 5.09, 5e-3),
    ("1,000 paths, one point: mean cost", S1000[0.01]["mean_cost"], 12.09, 5e-3),
    ("1,000 paths, two points: designs meeting 5%", pc(S1000[0.02]["meet"]), 100, 0.5),
    ("1,000 paths, two points: mean cost", S1000[0.02]["mean_cost"], 14.33, 5e-3),
    ("10,000 paths, no margin: designs meeting 5%", pc(S10000[0.0]["meet"]), 69, 0.5),
    ("10,000 paths, no margin: mean cost", S10000[0.0]["mean_cost"], 10.53, 5e-3),
    ("10,000 paths, quarter point: designs meeting 5%", pc(S10000[0.0025]["meet"]), 97, 0.5),
    ("10,000 paths, quarter point: mean true chance", pc(S10000[0.0025]["mean_true"]), 4.64, 5e-3),
    ("10,000 paths, quarter point: mean cost", S10000[0.0025]["mean_cost"], 10.97, 5e-3),
    ("kappa at a quarter-point margin (workbook)", BOUNDS[0]["kappa"], 6.69e-5, 5e-8),
    ("Chernoff bound, quarter point, 100,000 paths (workbook)", BOUNDS[0]["chernoff"], 0.0012, 5e-5),
    ("Chernoff bound, quarter point, 10,000 paths (workbook)", BOUNDS[1]["chernoff"], 0.51, 5e-3),
    ("Hoeffding bound, quarter point, 100,000 paths (workbook)", BOUNDS[0]["hoeffding"], 0.29, 5e-3),
    ("Hoeffding bound, quarter point, 10,000 paths (workbook)", BOUNDS[1]["hoeffding"], 0.88, 5e-3),
    ("kappa at one point (workbook)", BOUNDS[2]["kappa"], 0.00113, 5e-6),
    ("Chernoff bound, one point, 1,000 paths (workbook)", BOUNDS[2]["chernoff"], 0.32, 5e-3),
    ("Chernoff bound, two points, 1,000 paths (workbook)", BOUNDS[3]["chernoff"], 0.0076, 5e-5),
    # Worked Example 9.1, Step 3 and Table 9.3 (Panel 4)
    ("draw still ahead after quarter 3, % (workbook)", pc(AHEAD[2]), 29.9, 0.05),
    ("draw still ahead after quarter 6, % (workbook)", pc(AHEAD[5]), 3.7, 0.05),
    ("slow only, quarter 1: line", SLOW_ONLY[1]["line"], 170, 0),
    ("slow only, quarter 1: slowdown", SLOW_ONLY[1]["slow"], 1.00, 5e-3),
    ("slow only, quarter 1: trigger", SLOW_ONLY[1]["trigger"], 683, 0.5),
    ("slow only, quarter 1: share slowed", pc(SLOW_ONLY[1]["share"]), 98.5, 0.05),
    ("slow only, quarter 1: cost (workbook)", SLOW_ONLY[1]["cost"], 10.62, 5e-3),
    ("slow only, quarter 2: cost (workbook)", SLOW_ONLY[2]["cost"], 10.89, 5e-3),
    ("slow only, quarter 2: trigger", SLOW_ONLY[2]["trigger"], 615, 0.5),
    ("slow only, quarter 3: line", SLOW_ONLY[3]["line"], 175, 0),
    ("slow only, quarter 3: slowdown", SLOW_ONLY[3]["slow"], 4.00, 5e-3),
    ("slow only, quarter 3: share slowed", pc(SLOW_ONLY[3]["share"]), 64.3, 0.05),
    ("slow only, quarter 3: cost (workbook)", SLOW_ONLY[3]["cost"], 11.41, 5e-3),
    ("slow only, quarter 4: slowdown", SLOW_ONLY[4]["slow"], 6.50, 5e-3),
    ("slow only, quarter 4: cost (workbook)", SLOW_ONLY[4]["cost"], 12.31, 5e-3),
    ("slow only, quarters 5-6: no slowdown alone keeps it", sum(SLOW_ONLY[k] is not None for k in (5, 6)), 0, 0),
    ("commit and slow, quarter 1: cost (workbook)", BOTH[1]["cost"], 10.38, 5e-3),
    ("commit and slow, quarter 1: share slowed", pc(BOTH[1]["share"]), 44.9, 0.05),
    ("commit and slow, quarter 2: cost (workbook)", BOTH[2]["cost"], 10.30, 5e-3),
    ("commit and slow, quarter 2: trigger", BOTH[2]["trigger"], 478, 0.5),
    ("commit and slow, quarter 3: trigger", BOTH[3]["trigger"], 461, 0.5),
    ("commit and slow, quarter 3: share slowed", pc(BOTH[3]["share"]), 14.8, 0.05),
    ("commit and slow, quarter 3: cost (workbook)", BOTH[3]["cost"], 10.23, 5e-3),
    ("commit and slow, quarter 4: cost (workbook)", BOTH[4]["cost"], 10.31, 5e-3),
    ("commit and slow, quarter 5: cost (workbook)", BOTH[5]["cost"], 10.35, 5e-3),
    ("commit and slow, quarter 5: trigger", BOTH[5]["trigger"], 409, 0.5),
    ("commit and slow, quarter 6: cost (workbook)", BOTH[6]["cost"], 10.43, 5e-3),
    ("commit and slow, quarter 6: share slowed", pc(BOTH[6]["share"]), 7.6, 0.05),
    ("quarter 3, whole grid: the cheapest rule's cost", BEST3["cost"], 10.23, 5e-3),
    ("quarter 3, whole grid: committed delay", BEST3["base"], 0.55, 5e-3),
    ("quarter 3, whole grid: slowdown", BEST3["slow"], 2.50, 5e-3),
    ("quarter 3 rule saves against the package (workbook)", PKG[2] - BOTH[3]["cost"], 0.21, 5e-3),
    ("slow only, quarter 1: fresh chance", pc(ONE_FRESH["slow 1"][0]), 4.90, 5e-3),
    ("slow only, quarter 1: fresh cost", ONE_FRESH["slow 1"][1], 10.62, 5e-3),
    ("slow only, quarter 3: fresh chance", pc(ONE_FRESH["slow 3"][0]), 4.95, 5e-3),
    ("slow only, quarter 3: fresh cost", ONE_FRESH["slow 3"][1], 11.41, 5e-3),
    ("commit and slow, quarter 3: fresh chance", pc(ONE_FRESH["both 3"][0]), 4.96, 5e-3),
    ("commit and slow, quarter 3: fresh cost", ONE_FRESH["both 3"][1], 10.24, 5e-3),
    ("commit and slow, quarter 3: fresh share slowed", pc(ONE_FRESH["both 3"][2]), 14.9, 0.05),
    # Worked Example 9.1, Step 4 (Panel 5)
    ("every quarter: margin below the plan's path", EVERY["margin"], 72.6, 0.05),
    ("every quarter: share slowed", pc(EVERY["share"]), 33.5, 0.05),
    ("every quarter: slowed first at quarter 1", pc(slowed_by_q[0]), 7.3, 0.05),
    ("every quarter: slowed first at quarter 5", pc(slowed_by_q[4]), 5.3, 0.05),
    ("every quarter: contingent delay, quarters (workbook)", contingent, 0.156, 5e-4),
    ("every quarter: cost (workbook)", EVERY["cost"], 10.195, 5e-4),
    ("every quarter: saves against the package (workbook)", PKG[2] - EVERY["cost"], 0.25, 5e-3),
    ("every quarter: saves, percent (workbook)", 100 * (PKG[2] - EVERY["cost"]) / PKG[2], 2, 0.5),
    ("every quarter: fresh chance", pc(ev_fresh[0]), 4.95, 5e-3),
    ("every quarter: fresh cost", ev_fresh[1], 10.21, 5e-3),
    ("every quarter: the cheapest of its neighbours", near_best["cost"], 10.20, 5e-3),
    # Proposition 9.3 and Worked Example 9.1, Step 5 (Panel 6)
    ("clairvoyant: expected cost", CLAIR, 0.437, 5e-4),
    ("clairvoyant: paths given a package", pc(acting), 19.6, 0.05),
    ("clairvoyant: paths given a line", pc(lined), 16.3, 0.05),
    ("expected value of perfect information (workbook)", EVPI, 10.00, 5e-3),
    ("early warning: paths that see a closure", pc(w1), 25.9, 0.05),
    ("early warning, none ahead: line", WARN[1][1], 0, 0),
    ("early warning, none ahead: delay", WARN[1][2], 0.608, 5e-4),
    ("early warning, none ahead: cost (workbook)", WARN[1][0], 4.87, 5e-3),
    ("early warning, none ahead: chance", pc(WARN[1][3]), 4.75, 5e-3),
    ("early warning, closure ahead: line", WARN[2][1], 215, 0),
    ("early warning, closure ahead: delay", WARN[2][2], 0.504, 5e-4),
    ("early warning, closure ahead: cost (workbook)", WARN[2][0], 10.49, 5e-3),
    ("early warning, closure ahead: chance (workbook)", pc(WARN[2][3]), 5.71, 5e-3),
    ("early warning: expected cost (workbook)", WARN[0], 6.32, 5e-3),
    ("early warning: worth (workbook)", PKG[2] - WARN[0], 4.12, 5e-3),
    # Worked Example 9.1, Step 5: persistence (Panel 7)
    ("persistence 0.85: surprises only, some quarter", pc(noise85), 21.0, 0.05),
    ("persistence 0.85: committed line", PKG85[0], 165, 0),
    ("persistence 0.85: committed delay", PKG85[1], 2.90, 5e-3),
    ("persistence 0.85: committed cost (workbook)", PKG85[2], 28.18, 5e-3),
    ("persistence 0.85: committed, fresh chance", pc(pkg85_fresh), 5.04, 5e-3),
    ("persistence 0.85: every quarter, margin", EVERY85["margin"], 68.6, 0.05),
    ("persistence 0.85: every quarter, share slowed", pc(EVERY85["share"]), 38.3, 0.05),
    ("persistence 0.85: every quarter, cost", EVERY85["cost"], 25.88, 5e-3),
    ("persistence 0.85: every quarter, fresh chance", pc(ev85_fresh[0]), 5.01, 5e-3),
    ("persistence 0.85: every quarter, fresh cost", ev85_fresh[1], 25.96, 5e-3),
    ("persistence 0.85: reading cash saves (workbook)", PKG85[2] - EVERY85["cost"], 2.29, 5e-3),
    ("persistence 0.85: saves, percent (workbook)", 100 * (PKG85[2] - EVERY85["cost"]) / PKG85[2], 8.1, 0.05),
    # Theorems 9.1-9.2, Proposition 9.1 (Panels 8-9; Online Appendix 9.B.3)
    ("hiring: Riccati root p (workbook)", p0, 0.00090372, 5e-9),
    ("hiring: gain K, a year (workbook)", K0, 4.3379, 5e-5),
    ("hiring: departures cost sigma^2 p a year (workbook)", SIGMA ** 2 * p0, 0.20, 5e-3),
    ("hiring: departures cost in present value (workbook)", drag, 2.133, 5e-4),
    ("hiring: least cost with departures (workbook)", p0 * d0 ** 2 + drag, 11.171, 5e-4),
    ("hiring: beta = 2a + rho (workbook)", beta, 0.938194, 5e-7),
    ("caution, s = 0.10: gain (workbook)", Kc1, 4.1614, 5e-5),
    ("caution, s = 0.10: p (workbook)", pc1, 0.00093157, 5e-9),
    ("caution, s = 0.10: least cost (workbook)", pc1 * (d0 ** 2 + SIGMA ** 2 / rho), 11.515, 5e-4),
    ("caution, s = 0.20: gain (workbook)", Kc2, 3.6821, 5e-5),
    ("caution, s = 0.20: p (workbook)", pc2, 0.00101669, 5e-9),
    ("caution, s = 0.20: least cost (workbook)", pc2 * (d0 ** 2 + SIGMA ** 2 / rho), 12.567, 5e-4),
    ("certainty-equivalent gain at s = 0.10: cost (workbook)", gain_cost(K0, S_YIELD), 11.524, 5e-4),
    ("certainty-equivalent gain at s = 0.20: cost (workbook)", gain_cost(K0, 2 * S_YIELD), 12.730, 5e-4),
    ("ignoring the yield's uncertainty costs, s = 0.10 (workbook)", gain_cost(K0, S_YIELD) - gain_cost(Kc1, S_YIELD), 0.009, 5e-4),
    ("ignoring the yield's uncertainty costs, s = 0.20 (workbook)", gain_cost(K0, 2 * S_YIELD) - gain_cost(Kc2, 2 * S_YIELD),
     0.163, 5e-4),
    ("vigilance, c = 0.5: p (workbook)", pv, 0.00094136, 5e-9),
    ("vigilance, c = 0.5: gain (workbook)", Kv, 4.5185, 5e-5),
    ("robust, theta = 1: gain (workbook)", ROBUST[1][1], 4.659, 5e-4),
    ("robust, theta = 2: gain (workbook)", ROBUST[2][1], 5.070, 5e-4),
    ("robust, theta = 3: gain (workbook)", ROBUST[3][1], 5.624, 5e-4),
    ("robust, theta = 6: gain (workbook)", ROBUST[4][1], 11.283, 5e-4),
    ("robust, theta = 6.5: gain (workbook)", ROBUST[5][1], 20.621, 5e-4),
    ("robust, theta = 1: p x 1e4 (workbook)", 1e4 * ROBUST[1][0], 9.706, 5e-4),
    ("robust, theta = 6.5: p x 1e4 (workbook)", 1e4 * ROBUST[5][0], 42.960, 5e-4),
    ("alpha_theta turns negative at (workbook)", theta_flip, 6.4, 5e-3),
    ("breakdown point theta-bar (workbook)", theta_bar, 6.553, 5e-4),
    ("at the breakdown, p -> 2q/beta (workbook)", 2 * q / beta, 0.006822, 5e-7),
    ("at the breakdown, K -> (workbook)", b * 2 * q / beta / r, 32.7, 0.05),
    ("ledger, best rule, at 0 (workbook)", LEDGER["best"][0], 11.171, 5e-4),
    ("ledger, best rule, after 5 years (workbook)", LEDGER["best"][3], 11.171, 5e-4),
    ("ledger, cautious gain, after 1 year (workbook)", LEDGER["cautious"][1], 13.176, 5e-4),
    ("ledger, cautious gain, after 2 years (workbook)", LEDGER["cautious"][2], 13.287, 5e-4),
    ("ledger, cautious gain, after 5 years (workbook)", LEDGER["cautious"][3], 13.392, 5e-4),
    ("ledger, cautious gain, its limit (workbook)", rule_cost(2.0), 13.700, 5e-4),
]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"; ok += status == "PASS"
    fmt = ".8f" if 0 < abs(want) < 0.01 else ".4f"
    print(f"{name:62s} {got:14{fmt}}   book {want:12{fmt}}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 10 asks what can be guaranteed when only a bounded set of possible futures is declared, and
Chapter 11 what to do when several laws fit the record; risk sensitivity read as robustness (Proposition 9.1) is
the reading Chapter 11 builds on. AXIOM-09 runs the same models interactively (Lab 2.9); this notebook and the workbook are
its reproducible record.